In [347]:
%%capture
%load_ext autoreload
%autoreload 2

In [348]:
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.preprocessing import LabelEncoder
from recsys_pipeliner.recommendations.transformer import (
    SimilarityTransformer,
    UserItemMatrixTransformer,
)
from recsys_pipeliner.recommendations.recommender import (
    ItemBasedRecommender,
    UserBasedRecommender,
)
from IPython.display import display

In [349]:
# load test data
data_types = {"user_id": str, "item_id": str, "rating": np.float64}
user_item_ratings = pd.read_csv("../../tests/test_data/user_item_ratings_toy.csv", dtype=data_types)

display(user_item_ratings.head(3))

# encode the user/item ids
item_encoder = LabelEncoder()
user_encoder = LabelEncoder()

user_item_ratings["item_id"] = item_encoder.fit_transform(
    user_item_ratings["item_id"]
)
user_item_ratings["user_id"] = user_encoder.fit_transform(
    user_item_ratings["user_id"]
)

unique_users = pd.Series(user_encoder.classes_)
unique_items = pd.Series(item_encoder.classes_)

print("unique_users.shape", unique_users.shape)
print("unique_items.shape", unique_items.shape)

display(user_item_ratings.head(3))

# create the user/item matrix
user_item_matrix_transformer = UserItemMatrixTransformer()

user_item_matrix = user_item_matrix_transformer.transform(
    user_item_ratings.to_numpy(),
)

print("user_item_matrix.shape", user_item_matrix.shape)

# sanity check
users = user_item_ratings["user_id"].to_numpy().astype(int)
items = user_item_ratings["item_id"].to_numpy().astype(int)
ratings = user_item_ratings["rating"].to_numpy().astype(np.float32)
for user, item, rating in zip(users, items, ratings):
    assert user_item_matrix[user, item] == rating

,user_id,item_id,rating
0,U00001,I00024,0.8
1,U00001,I00013,0.6
2,U00001,I00005,1.0


unique_users.shape (12,)
unique_items.shape (24,)


,user_id,item_id,rating
0,0,23,0.8
1,0,12,0.6
2,0,4,1.0


user_item_matrix.shape (12, 24)


In [350]:
item_similarity_matrix_transformer = SimilarityTransformer()
item_similarity_matrix = item_similarity_matrix_transformer.transform(
    user_item_matrix.T
)

user_similarity_matrix_transformer = SimilarityTransformer()
user_similarity_matrix = user_similarity_matrix_transformer.transform(
    user_item_matrix
)

item_similarity_matrix.shape, user_similarity_matrix.shape

((24, 24), (12, 12))

In [351]:
# for use below
user_id = "U00003"
user_idx = user_encoder.transform([user_id])[0]
item_id = "I00003"
item_idx = item_encoder.transform([item_id])[0]
k=10
n=10

## Item-based prediction

Given a user_id and an item_id:

1. Get all items the user has rated
2. Sort by similarity to item_id
3. Get top k
4. Calculate weighted average rating
5. Return estimated score

In [352]:
# manual implementation

_, target_user_rated_items, target_user_ratings = sp.sparse.find(user_item_matrix[user_idx, :])

# remove the item_idx from the target_user_rated_items
target_user_rated_items = target_user_rated_items[target_user_rated_items != item_idx]
users_ratings = target_user_ratings[target_user_rated_items != item_idx]

# get the item similarities to item_idx
item_similarities = item_similarity_matrix[:, target_user_rated_items][item_idx].toarray().astype(np.float32).round(6)

# sort by similarity (desc) and get top k
top_k_mask = np.argsort(1 - item_similarities)[:k]
top_k_target_user_rated_items = target_user_rated_items[top_k_mask]
top_k_target_user_ratings = users_ratings[top_k_mask]
top_k_rated_item_similarities = item_similarities[top_k_mask]
users_unrated_items = np.setdiff1d(np.arange(item_similarity_matrix.shape[0]), top_k_target_user_rated_items)

# weighted average rating
predicted_item_based_rating = np.average(top_k_target_user_ratings, axis=0, weights=top_k_rated_item_similarities).astype(np.float32).round(6)
print(f"predicted rating for item {item_id} by user {user_id}", predicted_item_based_rating)

predicted rating for item I00003 by user U00003 0.839695


In [353]:
item_based_recommender = ItemBasedRecommender(k=k)
item_based_recommender.fit(user_item_matrix)

item_based_prediction_1 = item_based_recommender.predict(user_idx, item_idx)

assert item_based_prediction_1 == predicted_item_based_rating

In [354]:
# Reimplemented using the new classes

from recsys_pipeliner.algorithms.recommenders import ItemBasedCFRecommender

item_based_cf_recommender = ItemBasedCFRecommender(k=k, n=n)
item_based_cf_recommender.fit(user_item_matrix)

item_based_cf_predictions = item_based_cf_recommender.predict(np.array([[user_idx, item_idx]]))

assert item_based_cf_predictions[0] == predicted_item_based_rating == item_based_prediction_1

## User-based prediction

1. Get users who have rated item_id
2. Sort by similarity to user_id
3. Get top k
4. Get those users' rating of item_id
5. Calculate weighted average rating
6. Return estimated score

In [355]:
# manual implementation

_, all_users_with_ratings, all_users_ratings = sp.sparse.find(user_item_matrix[:, item_idx])

users = all_users_with_ratings[all_users_with_ratings != user_idx]
users_ratings = all_users_ratings[all_users_with_ratings != user_idx]

# get the similarities to user_id
_, similar_users, user_similarities = sp.sparse.find(user_similarity_matrix[user_idx, users])

# sort by similarity (desc) and get top k
top_k_mask = np.argsort(1 - user_similarities)[:k]
top_k_users = users[top_k_mask]
top_k_users_ratings = users_ratings[top_k_mask]
top_k_users_similarities = user_similarities[top_k_mask]

# weighted average rating
predicted_user_based_rating = np.average(top_k_users_ratings, axis=0, weights=top_k_users_similarities).astype(np.float32).round(6)
print(f"predicted rating for item {item_id} by user {user_id}", predicted_user_based_rating)

predicted rating for item I00003 by user U00003 0.859564


In [356]:
user_based_recommender = UserBasedRecommender(k=k)
user_based_recommender.fit(user_item_matrix)

user_based_prediction_1 = user_based_recommender.predict(user_idx, item_idx)

assert user_based_prediction_1 == predicted_user_based_rating

In [357]:
# Reimplement recommendations using the new classes
_, target_user_rated_items, _ = sp.sparse.find(user_item_matrix[user_idx, :])

# exclude items already rated and the item itself
candidates = np.setdiff1d(
    np.arange(item_similarity_matrix.shape[0]),
    np.concatenate([[item_idx], target_user_rated_items]),
)

item_similarity = item_similarity_matrix[item_idx, candidates]

_, item_indices, item_similarities = sp.sparse.find(item_similarity)
similar_items = candidates[item_indices]

sorter = np.argsort(1 - item_similarities, kind="stable")
recommendations = similar_items[sorter][:n]
defaults = np.full(n - recommendations.shape[0], -1)
item_based_recommendations = np.concatenate([recommendations, defaults])
print("item_based_recommendations", item_based_recommendations)

item_based_recommendations [21 17 11 16 20 22  9 14 -1 -1]


In [358]:
item_based_cf_recommendations = item_based_cf_recommender.recommend(
    np.array([[user_idx, item_idx]])
)[0]
print("item_based_cf_recommendations", item_based_cf_recommendations)
assert np.all(item_based_recommendations == item_based_cf_recommendations)

item_based_cf_recommendations [21 17 11 16 20 22  9 14 -1 -1]


In [359]:
# item_id only recommendations

item_based_cf_recommendations_unpersonalised = item_based_cf_recommender.recommend(
    np.array([item_idx])
)[0]
print("item_based_cf_recommendations_unpersonalised", item_based_cf_recommendations_unpersonalised)

item_based_cf_recommendations_unpersonalised [10 13  5 21 17 11 16  3 20 15]
